In [0]:
print("Ram")

**SparkSession :**

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
    .appName("reading_file")\
    .config("master","local[*]")\
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# spark.conf.set(<name-of-the-config-variable>, <value>)
# spark.conf.get(<name-of-the-config-variable>)


In [0]:
# spark.read.format("..")\
#     .option("key", "Value")\
#     .schema(...)\
#     .load(...)

# format : CSV , JSON, Parquet(default), Table, JDBC/ODBC
# option : inferScheam , Header, mode
# schema : manual schema
# load : path where data is residing
# Mode : 
#     1. FAILFAST : fail the execution if malformed record in dataset.
#     2. DROPMALFORMED : drop the malformed/corrupted record
#     3. PERMISSIVE : Default , Set NULL values to all corrupted fields.

In [0]:
flight_df = spark.read.format("csv")\
    .option("header",True)\
    .option("inferSchema","true")\
    .option("mode","FAILFAST")\
    .load("/Volumes/workspace/mde/testdata/flight_data.csv")

In [0]:
flight_df.printSchema()

**Schema in Spark :**
- How to create schema in spark ?
- What are other ways to creating it ?
- What is SrtuctField and StructType in schema?
- What if there is header in data ?

In [0]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType
# StructField : represents a column(field) in a schema.(<"col_nm">,<dataType>,<nullable>)
# StructType : represents schema of the DF which bascially is collection/list of fileds.

In [0]:
flight_schema = StructType([
    StructField("DEST_COUNTRY_NAME",StringType(),True),
    StructField("ORIGIN_COUNTRY_NAME",StringType(),True),
    StructField("count",IntegerType(),True)
])

# df = spark.createDataFrame(data = [(1,"Ram",25),(2,"Shyam",27),(3,"Meena",22)],schema = myschema)

In [0]:
flight_df_schema = spark.read.format("csv")\
    .option("header",False)\
    .schema(flight_schema)\
    .option("mode","PERMISSIVE")\
    .load("/Volumes/workspace/mde/testdata/flight_data.csv")

In [0]:
flight_df_schema.show(5)

In [0]:
flight_df_schema = spark.read.format("csv")\
    .option("header",False)\
    .option("skipRows",1) \
    .schema(flight_schema)\
    .option("mode","PERMISSIVE")\
    .load("/Volumes/workspace/mde/testdata/flight_data.csv")

In [0]:
flight_df_schema.show(5)

**Reading JSON File :** JavaScript Object Notation
- What is json data and how to read it in spark ?
- what if there are 3 keys in all line and 4 keys in one line ?
- What is multiline and line delimited json and which one work faster ?
- How to convert nested json into spark dataframe ?

In [0]:
line_Delimited_json_df = spark.read.format("json")\
    .option("mode","PERMISSIVE")\
    .load("/Volumes/workspace/mde/testdata/line_delimited_json.json")

# line Delimited Json : One JSON object per line.(more effecient in spark. "default")

In [0]:
line_Delimited_json_df.show()

In [0]:
extra_field_json_df = spark.read.format("json")\
    .option("mode","PERMISSIVE")\
    .load("/Volumes/workspace/mde/testdata/single_file_json_extrafields.json")



In [0]:
extra_field_json_df.show()

In [0]:
multiline_json_df = spark.read.format("json")\
    .option("mode","PERMISSIVE")\
    .option("multiline",True) \
    .load("/Volumes/workspace/mde/testdata/Multi_line_correct.json")

# multiline JSON : JSON array or nested document across multiple lines.

In [0]:
multiline_json_df.show()

In [0]:
corrupt_multiline_json_df = spark.read.format("json")\
    .option("mode","PERMISSIVE")\
    .option("multiline",True) \
    .load("/Volumes/workspace/mde/testdata/Multi_line_incorrect.json")

# multiline JSON : JSON array or nested document across multiple lines.

In [0]:
corrupt_multiline_json_df.show()

In [0]:
corrupt_df = spark.read.format("json")\
    .option("mode","PERMISSIVE")\
    .load("/Volumes/workspace/mde/testdata/corrupted_json.json")


In [0]:
corrupt_df.show(truncate = False)

**Reading parquet File :**
- What is parquet file format ?
- Why do we need parquet ?
- How to read parquet file ?
- What makes parquet default choice ?
- What compression technique is used ?
How to optimize the parquet file ?

In [0]:
# Parquet : Columnar based file format designed for storing large amounts of data efficiently.
#     Columnar based file format : data stored column by column (id -> 1,2,3,...;name -> "ram","Sita","Ajay",...)
#             -> stores all data in there specific dataType.and also store metadata in file.
#             -> mainly used in OLAP(write once read many)
#             -> low Cost to store(compressed) , less Time , high performance, less I/O
#             -> parquet,ORC
#     Row based file format : data stored row by row(1,"ram"/2,"Sita"/3,"Ajay")
#             -> Stores all the data as stringType(i.e.inferSchema is needed while reading the data which is costly)
#             -> used in OLTP( frequently writing/updating..)
#             -> CSV,AVRO,JSON

In [0]:
parquet_df = spark.read.format("parquet")\
.load("/Volumes/workspace/mde/testdata/part-r-00000-1a9822ba-b8fb-4d8e-844a-ea30d0801b9e.gz.parquet")

In [0]:
parquet_df.show(5)